In [21]:
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB

In [22]:
# 1. Load Data
df = pd.read_csv('ObesityDataSet_raw_and_data_sinthetic.csv')
#print(df.head())
X = df.drop('NObeyesdad', axis=1)
#print(X.head())
y = df['NObeyesdad']
print(y.head())

0          Normal_Weight
1          Normal_Weight
2          Normal_Weight
3     Overweight_Level_I
4    Overweight_Level_II
Name: NObeyesdad, dtype: object


In [23]:
# 2. Separate categorical and numerical columns
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()

In [24]:
# 3. Preprocessing: MinMaxScaler is REQUIRED for MultinomialNB
preprocessor = ColumnTransformer(
    transformers=[
        ('num', MinMaxScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ])

In [25]:
# 4. Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [26]:
# 5. Define the base models
# Note: probability=True is needed for SVC if you want to use voting='soft'
mnb = MultinomialNB()
svc = SVC(probability=True, random_state=42)
rf = RandomForestClassifier(random_state=42)

In [27]:
# 6. Create the Voting Classifier
ensemble = VotingClassifier(estimators=[
    ('mnb', mnb),
    ('svc', svc),
    ('rf', rf)
], voting='soft') # 'soft' averages the probabilities, 'hard' takes majority vote

In [28]:

# 7. Create a pipeline and Train
clf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor), 
    ('classifier', ensemble)
])

In [29]:
# 4. Define the Hyperparameter Grid
# You optimize the internal models by chaining the names: 'classifier__rf__n_estimators'
param_grid = {
    'classifier__mnb__alpha': [0.1, 0.5, 1.0, 5.0],                # MNB smoothing parameter
    'classifier__svc__C': [0.1, 1, 10],                            # SVC regularization
    'classifier__svc__kernel': ['linear', 'rbf'],                  # SVC shape of decision boundary
    'classifier__rf__n_estimators': [50, 100, 200],                # Number of trees in Random Forest
    'classifier__rf__max_depth': [None, 10, 20],                   # Max depth of the trees
    'classifier__voting': ['soft', 'hard']                         # Try both soft and hard voting
}

# 5. Run Randomized Search
# n_iter=10 means it will test 10 random combinations from the grid above
search = RandomizedSearchCV(
    clf_pipeline, 
    param_distributions=param_grid, 
    n_iter=10, 
    cv=3,           # 3-fold cross validation
    n_jobs=-1,      # Use all computer cores to speed it up
    random_state=42
)

print("Starting hyperparameter tuning... this may take a minute.")
search.fit(X_train, y_train)

# 6. Results
print("\nBest Parameters Found:")
for param, value in search.best_params_.items():
    print(f"{param}: {value}")

print(f"\nOptimized Accuracy on Test Set: {search.score(X_test, y_test) * 100:.2f}%")

Starting hyperparameter tuning... this may take a minute.

Best Parameters Found:
classifier__voting: soft
classifier__svc__kernel: linear
classifier__svc__C: 10
classifier__rf__n_estimators: 200
classifier__rf__max_depth: 10
classifier__mnb__alpha: 5.0

Optimized Accuracy on Test Set: 94.56%
